# Phase 6 DAEAC FCBA RR Late-Fusion NSV MITBIH Sqrt-Weight Pretrain

Pretrains on natural MITBIH train with sqrt class weighting, selects the best checkpoint on MITBIH validation, then evaluates on INCART and SVDB.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIG = 'configs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw.yaml'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'

## 1. Clone Repo and Install

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), ECG
os.chdir(ECG)
!pip install -q -r requirements.txt
!python scripts/check_repo.py

## 2. Copy NSV RR Bundle

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV bundle, found {candidate_dirs}'

DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

required = ('mitbih_train.npz', 'mitbih_val.npz', 'incart_train.npz', 'incart_val.npz', 'incart_test.npz', 'svdb_test.npz')
for name in required:
    path = DEST / name
    assert path.exists(), f'Missing {path}.'
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['rr_features'].shape)

with np.load(DEST / 'mitbih_train.npz', allow_pickle=True) as data:
    labels, counts = np.unique(data['y'], return_counts=True)
    print('mitbih_train counts:', dict(zip(labels.tolist(), counts.tolist())))
    print('aug_total:', int(data['is_augmented'].sum()))
!ls -lh data/processed/phase6_daeac_record_splits_nsv | head

## 3. Validate

In [ ]:
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', CONFIG], check=True)

## 4. Full MITBIH Sqrt-Weight Pretrain

In [ ]:
subprocess.run([
    sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
    '--config', CONFIG,
], check=True)

## 5. Inspect Best Checkpoint

In [ ]:
summary_path = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw/metrics/daeac_base_train_summary.json')
assert summary_path.exists(), summary_path
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
best_checkpoint = Path(summary['best_checkpoint'])
assert best_checkpoint.exists(), best_checkpoint
print('BEST_CHECKPOINT =', best_checkpoint)

## 6. Evaluate on INCART and SVDB

In [ ]:
for dataset in ('target', 'svdb'):
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
        '--config', CONFIG,
        '--checkpoint', str(best_checkpoint),
        '--method-name', 'daeac_fcba_rr_nsv_mitbih_sqrtw_base_best',
        '--dataset', dataset,
    ], check=True)

for path in sorted(Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw/metrics').glob('daeac_fcba_rr_nsv_mitbih_sqrtw_base_best_*_metrics.json')):
    print('\n', path)
    print(path.read_text())

## 7. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw_pretrain_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw